# Module 3: Exploratory Data Analysis with Temporal Discipline

## Learning Objectives

By the end of this module, you will:
1. Understand how to conduct EDA that respects temporal boundaries
2. Visualize data availability and quality over time
3. Compare different data vintages to understand revision impacts
4. Detect structural breaks and regime changes in commodity markets
5. Identify and visualize data quality issues

## Why Temporal Discipline Matters in EDA

Traditional EDA often uses "God's eye view" - analyzing the entire dataset as if all information was always available. In trading, this creates **look-ahead bias** that invalidates backtests.

**Key Principle**: Every analysis must respect what was knowable at each point in time.

In [ ]:
# Setup and imports
import sys
sys.path.append('../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Fair Value Toolkit imports
from src.fair_value_toolkit import (
    PointInTimeDataFrame,
    PointInTimeRecord,
    create_lagged_features,
    create_rolling_features,
)

from data.data_fetchers import (
    create_sample_dataset,
    CommodityDataFetcher,
)

# Plotting configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("Environment ready!")

## Part 1: Point-in-Time Exploratory Analysis

### 1.1 Creating Sample Dataset with Multiple Vintages

In [ ]:
# Create sample dataset
data = create_sample_dataset(
    commodity='crude_oil',
    start_date='2020-01-01',
    end_date='2023-12-31',
    include_revisions=True,  # Include data revisions
    include_delays=True       # Simulate publication delays
)

print(f"Dataset shape: {data.shape}")
print(f"\nColumns: {list(data.columns)}")
print(f"\nDate range: {data.index.min()} to {data.index.max()}")
data.head()

### 1.2 Understanding Point-in-Time Data Structure

Point-in-time data tracks:
- **observation_date**: When the event occurred
- **publication_date**: When we first learned about it
- **vintage**: Which revision of the data
- **value**: The actual data point

In [ ]:
# Convert to PointInTimeDataFrame for proper temporal handling
pit_data = PointInTimeDataFrame(data)

# Examine a specific observation across vintages
obs_date = pd.Timestamp('2022-06-15')
pit_data.get_all_vintages(obs_date, variable='inventory')

### 1.3 Point-in-Time Summary Statistics

Calculate statistics as they would have appeared at each point in time.

In [ ]:
def point_in_time_summary(pit_df, as_of_date, lookback_days=365):
    """
    Calculate summary statistics as of a specific date.
    Only uses data that was available at that time.
    """
    # Get data as it existed at as_of_date
    historical = pit_df.as_of(as_of_date)
    
    # Look back specified period
    start_date = as_of_date - timedelta(days=lookback_days)
    subset = historical[historical.index >= start_date]
    
    summary = pd.DataFrame()
    for col in subset.select_dtypes(include=[np.number]).columns:
        summary[col] = {
            'count': subset[col].count(),
            'mean': subset[col].mean(),
            'std': subset[col].std(),
            'min': subset[col].min(),
            '25%': subset[col].quantile(0.25),
            '50%': subset[col].quantile(0.50),
            '75%': subset[col].quantile(0.75),
            'max': subset[col].max(),
        }
    
    return summary.T

# Example: Summary statistics as of June 2022
as_of = pd.Timestamp('2022-06-30')
summary = point_in_time_summary(pit_data, as_of)
print(f"Summary Statistics as of {as_of.date()}")
print(f"Using data from previous 365 days\n")
summary

### 1.4 Visualizing Data Availability Over Time

Understanding when data becomes available is crucial for backtesting.

In [ ]:
def plot_data_availability(df, window='M'):
    """
    Visualize data completeness over time.
    """
    # Resample to desired frequency and count non-null values
    availability = df.notna().resample(window).mean() * 100
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    for col in availability.columns:
        ax.plot(availability.index, availability[col], label=col, marker='o', markersize=3)
    
    ax.axhline(y=95, color='green', linestyle='--', alpha=0.5, label='95% threshold')
    ax.axhline(y=80, color='orange', linestyle='--', alpha=0.5, label='80% threshold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Data Availability (%)')
    ax.set_title('Data Completeness Over Time')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_data_availability(data)

### 1.5 Rolling Statistics with Proper Lookbacks

Calculate rolling statistics that respect temporal boundaries.

In [ ]:
def calculate_expanding_statistics(pit_df, variable, as_of_dates):
    """
    Calculate expanding window statistics at multiple points in time.
    This shows how our understanding of the data evolves.
    """
    results = []
    
    for date in as_of_dates:
        # Get data as it existed at this date
        historical = pit_df.as_of(date)
        
        if variable in historical.columns:
            series = historical[variable].dropna()
            if len(series) > 0:
                results.append({
                    'date': date,
                    'mean': series.mean(),
                    'std': series.std(),
                    'min': series.min(),
                    'max': series.max(),
                    'count': len(series)
                })
    
    return pd.DataFrame(results).set_index('date')

# Calculate statistics at quarterly intervals
dates = pd.date_range('2020-03-31', '2023-12-31', freq='Q')
expanding_stats = calculate_expanding_statistics(pit_data, 'price', dates)

# Visualize evolving statistics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

expanding_stats['mean'].plot(ax=axes[0, 0], marker='o', title='Evolving Mean Price')
axes[0, 0].set_ylabel('Price ($)')
axes[0, 0].grid(True, alpha=0.3)

expanding_stats['std'].plot(ax=axes[0, 1], marker='o', title='Evolving Volatility', color='orange')
axes[0, 1].set_ylabel('Std Dev ($)')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(expanding_stats.index, expanding_stats['min'], marker='o', label='Min', color='red')
axes[1, 0].plot(expanding_stats.index, expanding_stats['max'], marker='o', label='Max', color='green')
axes[1, 0].set_title('Evolving Range')
axes[1, 0].set_ylabel('Price ($)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

expanding_stats['count'].plot(ax=axes[1, 1], marker='o', title='Data Points Available', color='purple')
axes[1, 1].set_ylabel('Count')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 2: Vintage Comparison

### 2.1 Understanding Data Revisions

Many commodity market data series are revised:
- **Inventory data**: Initial estimates vs. final counts
- **Production data**: Preliminary vs. revised figures
- **Economic indicators**: First release vs. subsequent revisions

These revisions can significantly impact models trained on "final" data.

In [ ]:
def simulate_data_revisions(df, variable, revision_prob=0.3, revision_magnitude=0.05):
    """
    Simulate data revisions for demonstration.
    
    In practice, this data would come from actual vintage datasets.
    """
    np.random.seed(42)
    
    vintages = pd.DataFrame()
    vintages['original'] = df[variable].copy()
    
    # First revision (1 month later)
    revision_mask = np.random.random(len(df)) < revision_prob
    revision_factor = 1 + np.random.randn(len(df)) * revision_magnitude
    vintages['revision_1'] = df[variable].copy()
    vintages.loc[revision_mask, 'revision_1'] *= revision_factor[revision_mask]
    
    # Second revision (3 months later)
    revision_mask = np.random.random(len(df)) < revision_prob * 0.5
    revision_factor = 1 + np.random.randn(len(df)) * revision_magnitude * 0.5
    vintages['revision_2'] = vintages['revision_1'].copy()
    vintages.loc[revision_mask, 'revision_2'] *= revision_factor[revision_mask]
    
    return vintages

# Simulate revisions for inventory data
inventory_vintages = simulate_data_revisions(data, 'inventory', revision_prob=0.4, revision_magnitude=0.08)
inventory_vintages.head(10)

### 2.2 Side-by-Side Vintage Comparison

In [ ]:
# Visualize different vintages
fig, ax = plt.subplots(figsize=(14, 6))

sample_period = slice('2022-01-01', '2022-06-30')
inventory_vintages.loc[sample_period, 'original'].plot(
    ax=ax, label='Original (First Release)', linewidth=2, alpha=0.7
)
inventory_vintages.loc[sample_period, 'revision_1'].plot(
    ax=ax, label='Revision 1 (1 month later)', linewidth=2, alpha=0.7, linestyle='--'
)
inventory_vintages.loc[sample_period, 'revision_2'].plot(
    ax=ax, label='Revision 2 (Final)', linewidth=2, alpha=0.7, linestyle=':'
)

ax.set_title('Inventory Data Across Vintages')
ax.set_xlabel('Date')
ax.set_ylabel('Inventory (million barrels)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 2.3 Revision Magnitude Statistics

In [ ]:
def analyze_revisions(vintages):
    """
    Calculate statistics on data revisions.
    """
    results = {}
    
    # First revision
    rev1_pct_change = (vintages['revision_1'] - vintages['original']) / vintages['original'] * 100
    results['revision_1'] = {
        'mean_pct_change': rev1_pct_change.mean(),
        'std_pct_change': rev1_pct_change.std(),
        'max_abs_change': rev1_pct_change.abs().max(),
        'pct_revised': (rev1_pct_change != 0).sum() / len(rev1_pct_change) * 100,
    }
    
    # Second revision
    rev2_pct_change = (vintages['revision_2'] - vintages['revision_1']) / vintages['revision_1'] * 100
    results['revision_2'] = {
        'mean_pct_change': rev2_pct_change.mean(),
        'std_pct_change': rev2_pct_change.std(),
        'max_abs_change': rev2_pct_change.abs().max(),
        'pct_revised': (rev2_pct_change != 0).sum() / len(rev2_pct_change) * 100,
    }
    
    # Total revision from original to final
    total_pct_change = (vintages['revision_2'] - vintages['original']) / vintages['original'] * 100
    results['total'] = {
        'mean_pct_change': total_pct_change.mean(),
        'std_pct_change': total_pct_change.std(),
        'max_abs_change': total_pct_change.abs().max(),
        'pct_revised': (total_pct_change != 0).sum() / len(total_pct_change) * 100,
    }
    
    return pd.DataFrame(results).T

revision_stats = analyze_revisions(inventory_vintages)
print("Revision Statistics:\n")
revision_stats

### 2.4 Distribution of Revisions

In [ ]:
# Calculate revision changes
rev1_change = (inventory_vintages['revision_1'] - inventory_vintages['original']) / inventory_vintages['original'] * 100
rev2_change = (inventory_vintages['revision_2'] - inventory_vintages['revision_1']) / inventory_vintages['revision_1'] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of first revisions
axes[0].hist(rev1_change.dropna(), bins=50, alpha=0.7, edgecolor='black')
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Revision (%)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of First Revisions')
axes[0].grid(True, alpha=0.3)

# Histogram of second revisions
axes[1].hist(rev2_change.dropna(), bins=50, alpha=0.7, edgecolor='black', color='orange')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Revision (%)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Second Revisions')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 3: Detecting Structural Breaks and Regime Changes

### 3.1 CUSUM Test for Structural Breaks

The CUSUM (Cumulative Sum) test detects changes in the mean of a time series.

In [ ]:
def cusum_test(series, threshold=1.0):
    """
    Perform CUSUM test for structural breaks.
    
    Parameters
    ----------
    series : pd.Series
        Time series to test
    threshold : float
        Threshold for detecting breaks (in standard deviations)
    
    Returns
    -------
    cusum : pd.Series
        CUSUM statistic over time
    breaks : list
        Dates where breaks were detected
    """
    # Calculate cumulative sum of deviations from mean
    mean = series.mean()
    std = series.std()
    
    deviations = (series - mean) / std
    cusum = deviations.cumsum()
    
    # Detect breaks where CUSUM crosses threshold
    upper_threshold = threshold * np.sqrt(len(series))
    lower_threshold = -upper_threshold
    
    breaks = []
    for i in range(1, len(cusum)):
        if (cusum.iloc[i] > upper_threshold and cusum.iloc[i-1] <= upper_threshold) or \
           (cusum.iloc[i] < lower_threshold and cusum.iloc[i-1] >= lower_threshold):
            breaks.append(cusum.index[i])
    
    return cusum, breaks

# Test for structural breaks in price
price_cusum, price_breaks = cusum_test(data['price'].dropna(), threshold=0.8)

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Original series
data['price'].plot(ax=axes[0], label='Price', linewidth=1.5)
for break_date in price_breaks:
    axes[0].axvline(x=break_date, color='red', linestyle='--', alpha=0.7)
axes[0].set_ylabel('Price ($)')
axes[0].set_title('Oil Price with Detected Structural Breaks')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# CUSUM statistic
price_cusum.plot(ax=axes[1], label='CUSUM', linewidth=1.5, color='blue')
threshold_val = 0.8 * np.sqrt(len(price_cusum))
axes[1].axhline(y=threshold_val, color='red', linestyle='--', alpha=0.5, label='Upper threshold')
axes[1].axhline(y=-threshold_val, color='red', linestyle='--', alpha=0.5, label='Lower threshold')
axes[1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
for break_date in price_breaks:
    axes[1].axvline(x=break_date, color='red', linestyle='--', alpha=0.7)
axes[1].set_ylabel('CUSUM Statistic')
axes[1].set_xlabel('Date')
axes[1].set_title('CUSUM Test Statistic')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Detected {len(price_breaks)} structural breaks at:")
for break_date in price_breaks:
    print(f"  - {break_date.date()}")

### 3.2 Rolling Correlation Analysis

Relationships between variables can change over time. Rolling correlations reveal these regime changes.

In [ ]:
def rolling_correlation(df, var1, var2, window=90):
    """
    Calculate rolling correlation between two variables.
    """
    return df[var1].rolling(window=window).corr(df[var2])

# Calculate rolling correlations
corr_price_inventory = rolling_correlation(data, 'price', 'inventory', window=90)
corr_price_production = rolling_correlation(data, 'price', 'production', window=90)

# Visualize
fig, ax = plt.subplots(figsize=(14, 6))

corr_price_inventory.plot(ax=ax, label='Price vs Inventory', linewidth=2)
corr_price_production.plot(ax=ax, label='Price vs Production', linewidth=2)

ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax.axhline(y=0.5, color='green', linestyle='--', alpha=0.3, label='Strong positive')
ax.axhline(y=-0.5, color='red', linestyle='--', alpha=0.3, label='Strong negative')

ax.set_xlabel('Date')
ax.set_ylabel('Correlation')
ax.set_title('Rolling 90-Day Correlations (Regime Detection)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-1, 1)

plt.tight_layout()
plt.show()

### 3.3 Regime Identification with Volatility Clustering

Commodity markets often exhibit distinct volatility regimes.

In [ ]:
def identify_volatility_regimes(returns, short_window=20, long_window=60):
    """
    Identify high/low volatility regimes.
    """
    # Calculate rolling volatility
    rolling_vol = returns.rolling(window=short_window).std()
    vol_ma = rolling_vol.rolling(window=long_window).mean()
    vol_std = rolling_vol.rolling(window=long_window).std()
    
    # Classify regimes
    regimes = pd.Series(index=returns.index, dtype='object')
    regimes[rolling_vol > vol_ma + vol_std] = 'High Volatility'
    regimes[rolling_vol < vol_ma - vol_std] = 'Low Volatility'
    regimes[(rolling_vol >= vol_ma - vol_std) & (rolling_vol <= vol_ma + vol_std)] = 'Normal'
    
    return rolling_vol, vol_ma, regimes

# Calculate returns
returns = data['price'].pct_change() * 100  # Percentage returns

# Identify regimes
rolling_vol, vol_ma, regimes = identify_volatility_regimes(returns)

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Price with regime backgrounds
data['price'].plot(ax=axes[0], linewidth=1.5, color='black')

# Color background by regime
high_vol_periods = regimes == 'High Volatility'
low_vol_periods = regimes == 'Low Volatility'

for i in range(len(regimes)-1):
    if high_vol_periods.iloc[i]:
        axes[0].axvspan(regimes.index[i], regimes.index[i+1], alpha=0.2, color='red')
    elif low_vol_periods.iloc[i]:
        axes[0].axvspan(regimes.index[i], regimes.index[i+1], alpha=0.2, color='green')

axes[0].set_ylabel('Price ($)')
axes[0].set_title('Price with Volatility Regimes')
axes[0].grid(True, alpha=0.3)

# Rolling volatility
rolling_vol.plot(ax=axes[1], label='Rolling Volatility', linewidth=1.5)
vol_ma.plot(ax=axes[1], label='Volatility MA', linewidth=2, linestyle='--', color='blue')
(vol_ma + rolling_vol.rolling(window=60).std()).plot(
    ax=axes[1], label='Upper Band', linewidth=1, linestyle=':', color='red'
)
(vol_ma - rolling_vol.rolling(window=60).std()).plot(
    ax=axes[1], label='Lower Band', linewidth=1, linestyle=':', color='green'
)

axes[1].set_ylabel('Volatility (%)')
axes[1].set_xlabel('Date')
axes[1].set_title('Rolling Volatility and Regime Bands')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Regime statistics
print("\nVolatility Regime Distribution:")
print(regimes.value_counts())

## Part 4: Visualizing Data Quality Over Time

### 4.1 Missing Data Patterns

In [ ]:
def plot_missing_data_heatmap(df, freq='W'):
    """
    Create a heatmap showing missing data patterns.
    """
    # Resample to desired frequency and check for missing data
    missing_data = df.isna().resample(freq).mean() * 100
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Create heatmap
    sns.heatmap(
        missing_data.T,
        cmap='RdYlGn_r',
        cbar_kws={'label': 'Missing Data (%)'},
        ax=ax,
        vmin=0,
        vmax=100
    )
    
    ax.set_xlabel('Date')
    ax.set_ylabel('Variable')
    ax.set_title('Data Completeness Heatmap (Weekly)')
    
    # Rotate x-axis labels for readability
    plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()

plot_missing_data_heatmap(data)

### 4.2 Publication Delay Tracking

In [ ]:
def simulate_publication_delays(df, mean_delay_days=7, std_delay_days=3):
    """
    Simulate publication delays for data.
    
    In production, this would come from actual publication date metadata.
    """
    np.random.seed(42)
    
    delays = pd.DataFrame(index=df.index)
    
    for col in df.columns:
        # Simulate delay in days
        delay_days = np.random.normal(mean_delay_days, std_delay_days, len(df))
        delay_days = np.maximum(delay_days, 1)  # At least 1 day
        delays[col] = delay_days
    
    return delays

# Simulate delays
delays = simulate_publication_delays(data)

# Visualize delay distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot of delays by variable
delays.boxplot(ax=axes[0])
axes[0].set_ylabel('Delay (days)')
axes[0].set_title('Publication Delay Distribution by Variable')
axes[0].grid(True, alpha=0.3)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')

# Time series of average delay
avg_delay = delays.resample('M').mean().mean(axis=1)
avg_delay.plot(ax=axes[1], marker='o', linewidth=2)
axes[1].set_ylabel('Average Delay (days)')
axes[1].set_xlabel('Date')
axes[1].set_title('Average Publication Delay Over Time')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.3 Data Completeness Score

In [ ]:
def calculate_data_quality_score(df, weights=None):
    """
    Calculate a composite data quality score.
    
    Considers:
    - Completeness (% non-null)
    - Freshness (recency of updates)
    - Consistency (absence of outliers)
    """
    if weights is None:
        weights = {'completeness': 0.4, 'freshness': 0.3, 'consistency': 0.3}
    
    scores = pd.DataFrame()
    
    for col in df.select_dtypes(include=[np.number]).columns:
        series = df[col]
        
        # Completeness score (0-100)
        completeness = (series.notna().sum() / len(series)) * 100
        
        # Freshness score (based on recency - simplified)
        # Higher score for more recent data
        freshness = 100  # Simplified for this example
        
        # Consistency score (based on outlier detection)
        if series.notna().sum() > 0:
            z_scores = np.abs((series - series.mean()) / series.std())
            outlier_pct = (z_scores > 3).sum() / len(series) * 100
            consistency = max(0, 100 - outlier_pct * 10)  # Penalize outliers
        else:
            consistency = 0
        
        # Composite score
        composite = (
            weights['completeness'] * completeness +
            weights['freshness'] * freshness +
            weights['consistency'] * consistency
        )
        
        scores[col] = {
            'completeness': completeness,
            'freshness': freshness,
            'consistency': consistency,
            'composite': composite
        }
    
    return scores.T

quality_scores = calculate_data_quality_score(data)

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))

quality_scores[['completeness', 'consistency', 'composite']].plot(
    kind='bar',
    ax=ax,
    width=0.8
)

ax.axhline(y=80, color='orange', linestyle='--', alpha=0.5, label='Good threshold')
ax.axhline(y=95, color='green', linestyle='--', alpha=0.5, label='Excellent threshold')
ax.set_ylabel('Score (0-100)')
ax.set_xlabel('Variable')
ax.set_title('Data Quality Scores by Variable')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

print("\nData Quality Scores:\n")
print(quality_scores)

### 4.4 Comprehensive Data Quality Dashboard

In [ ]:
def create_data_quality_dashboard(df):
    """
    Create a comprehensive data quality dashboard.
    """
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)
    
    # 1. Completeness over time
    ax1 = fig.add_subplot(gs[0, :])
    completeness = df.notna().resample('M').mean() * 100
    completeness.plot(ax=ax1, marker='o', markersize=4)
    ax1.axhline(y=95, color='green', linestyle='--', alpha=0.5)
    ax1.set_ylabel('Completeness (%)')
    ax1.set_title('Data Completeness Over Time (Monthly)')
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax1.grid(True, alpha=0.3)
    
    # 2. Missing data counts
    ax2 = fig.add_subplot(gs[1, 0])
    missing_counts = df.isna().sum().sort_values(ascending=True)
    missing_counts.plot(kind='barh', ax=ax2)
    ax2.set_xlabel('Number of Missing Values')
    ax2.set_title('Missing Data by Variable')
    ax2.grid(True, alpha=0.3, axis='x')
    
    # 3. Data availability matrix
    ax3 = fig.add_subplot(gs[1, 1])
    sample_data = df.iloc[::30]  # Sample every 30 rows for visibility
    sns.heatmap(
        sample_data.notna().T,
        cmap='RdYlGn',
        cbar_kws={'label': 'Data Available'},
        ax=ax3,
        xticklabels=False
    )
    ax3.set_title('Data Availability Pattern (Sampled)')
    
    # 4. Rolling data quality score
    ax4 = fig.add_subplot(gs[2, :])
    rolling_completeness = df.notna().rolling(window=30).mean().mean(axis=1) * 100
    rolling_completeness.plot(ax=ax4, linewidth=2, color='blue')
    ax4.fill_between(
        rolling_completeness.index,
        rolling_completeness,
        95,
        where=(rolling_completeness >= 95),
        alpha=0.3,
        color='green',
        label='Excellent'
    )
    ax4.fill_between(
        rolling_completeness.index,
        rolling_completeness,
        80,
        where=((rolling_completeness >= 80) & (rolling_completeness < 95)),
        alpha=0.3,
        color='orange',
        label='Good'
    )
    ax4.axhline(y=95, color='green', linestyle='--', alpha=0.5)
    ax4.axhline(y=80, color='orange', linestyle='--', alpha=0.5)
    ax4.set_ylabel('Quality Score (%)')
    ax4.set_xlabel('Date')
    ax4.set_title('Rolling 30-Day Average Data Quality')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.suptitle('Data Quality Dashboard', fontsize=16, fontweight='bold', y=0.995)
    plt.show()

create_data_quality_dashboard(data)

## Summary and Key Takeaways

### What We've Learned

1. **Point-in-Time Analysis**:
   - Always calculate statistics as of specific dates
   - Use proper lookback windows that respect temporal boundaries
   - Visualize data availability to understand limitations

2. **Vintage Comparison**:
   - Data revisions are common and can be substantial
   - Models trained on final data may not reflect real-time performance
   - Track revision patterns to understand data quality

3. **Structural Breaks**:
   - Use CUSUM tests to detect regime changes
   - Rolling correlations reveal changing relationships
   - Volatility clustering indicates distinct market regimes

4. **Data Quality**:
   - Track completeness, freshness, and consistency
   - Visualize missing data patterns over time
   - Monitor publication delays that affect data availability

### Critical Questions to Ask

Before using any dataset for modeling:
- When was each data point first available?
- Has the data been revised since initial publication?
- Are there systematic patterns in missing data?
- Have relationships between variables changed over time?
- Are there distinct market regimes in the data?

### Next Steps

In the next module, we'll explore specific commodity data sources and learn how to:
- Access free and premium data sources
- Integrate multiple APIs
- Assess and improve data quality
- Handle common data issues with temporal validity

## Exercises

1. **Point-in-Time Summary**: Create a function that generates summary statistics for multiple as-of dates and plots how key metrics evolve over time.

2. **Revision Impact**: Analyze how using different data vintages affects a simple trading signal (e.g., inventory z-score).

3. **Custom CUSUM**: Modify the CUSUM test to detect breaks in variance instead of mean.

4. **Quality Score**: Enhance the data quality score to include additional factors like:
   - Update frequency
   - Historical revision magnitude
   - Cross-source validation

5. **Regime Classification**: Use the volatility regime identification to create a trading rule that adjusts position sizing based on current regime.